BITS PILANI - Machine Learning - S1-26_AIMLZG565 - ML Assignment 2

- Implement multiple classification models
- Build an interactive Streamlit web application to demonstrate your models
- Deploy the app on Streamlit Community Cloud (FREE)
- Share clickable links for evaluatio

Student Name : Malasani Haritha

Student Id   : 2026ak05008


##1. Load the Dataset

In [89]:
# Import pandas – the library used to work with tables of data
import pandas as pd

# Link to the raw CSV file stored in my GitHub repository
url = "https://raw.githubusercontent.com/HarithaAIML/ml-classification-streamlit-app/main/in-vehicle-coupon-recommendation.csv"

# Read the CSV file into a DataFrame (a table) called df
df = pd.read_csv(url)

# Print the size of the table as (rows, columns)
print(df.shape)

# Show the first 5 rows to get a feel for the data
df.head()

(12684, 26)


,destination,passanger,weather,temperature,time,coupon,expiration,gender,age,maritalStatus,...,CoffeeHouse,CarryAway,RestaurantLessThan20,Restaurant20To50,toCoupon_GEQ5min,toCoupon_GEQ15min,toCoupon_GEQ25min,direction_same,direction_opp,Y
0,No Urgent Place,Alone,Sunny,55,2PM,Restaurant(<20),1d,Female,21,Unmarried partner,...,never,NaN,4~8,1~3,1,0,0,0,1,1
1,No Urgent Place,Friend(s),Sunny,80,10AM,Coffee House,2h,Female,21,Unmarried partner,...,never,NaN,4~8,1~3,1,0,0,0,1,0
2,No Urgent Place,Friend(s),Sunny,80,10AM,Carry out & Take away,2h,Female,21,Unmarried partner,...,never,NaN,4~8,1~3,1,1,0,0,1,1
3,No Urgent Place,Friend(s),Sunny,80,2PM,Coffee House,2h,Female,21,Unmarried partner,...,never,NaN,4~8,1~3,1,1,0,0,1,0
4,No Urgent Place,Friend(s),Sunny,80,2PM,Coffee House,1d,Female,21,Unmarried partner,...,never,NaN,4~8,1~3,1,1,0,0,1,0


## 2. Explore the Data
### 2.1 Column types

In [90]:
# Show each column's name, number of non-empty values, and data type

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12684 entries, 0 to 12683
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   destination           12684 non-null  object
 1   passanger             12684 non-null  object
 2   weather               12684 non-null  object
 3   temperature           12684 non-null  int64 
 4   time                  12684 non-null  object
 5   coupon                12684 non-null  object
 6   expiration            12684 non-null  object
 7   gender                12684 non-null  object
 8   age                   12684 non-null  object
 9   maritalStatus         12684 non-null  object
 10  has_children          12684 non-null  int64 
 11  education             12684 non-null  object
 12  occupation            12684 non-null  object
 13  income                12684 non-null  object
 14  car                   108 non-null    object
 15  Bar                   12577 non-null

### 2.2 Missing values

In [91]:
# Count how many empty (missing) values each column has
# This tells us which columns need cleaning before training models
df.isnull().sum()

,0
destination,0
passanger,0
weather,0
temperature,0
time,0
coupon,0
expiration,0
gender,0
age,0
maritalStatus,0


### 2.3 Class balance (target column Y)

In [92]:
# Count how many people accepted the coupon (1) vs did not accept (0)
print(df['Y'].value_counts())

# Same counts as percentages (normalize=True gives fractions that add up to 1)
# This shows whether the two classes are balanced or one is much bigger
print(df['Y'].value_counts(normalize=True))


Y
1    7210
0    5474
Name: count, dtype: int64
Y
1    0.568433
0    0.431567
Name: proportion, dtype: float64


### 2.4 Unique values per column

In [93]:
# Count how many different values each column has
# toCoupon_GEQ5min has only 1 value, so it cannot help prediction
print(df.nunique())

# Check that direction_opp is always the opposite of direction_same
# If every row adds up to 1, one of the two columns is redundant
print((df['direction_same'] + df['direction_opp']).unique())

destination              3
passanger                4
weather                  3
temperature              3
time                     5
coupon                   5
expiration               2
gender                   2
age                      8
maritalStatus            5
has_children             2
education                6
occupation              25
income                   9
car                      5
Bar                      5
CoffeeHouse              5
CarryAway                5
RestaurantLessThan20     5
Restaurant20To50         5
toCoupon_GEQ5min         1
toCoupon_GEQ15min        2
toCoupon_GEQ25min        2
direction_same           2
direction_opp            2
Y                        2
dtype: int64
[1]


## 3. Data Preprocessing
### 3.1 Removing unhelpful columns and duplicate rows

In [94]:
# Drop columns that cannot help prediction:
# - car: about 99% of values are missing
# - toCoupon_GEQ5min: has the same value (1) in every row
# - direction_opp: exact opposite of direction_same, so it adds no new information
df = df.drop(columns=['car', 'toCoupon_GEQ5min', 'direction_opp'])

# Check how many rows are exact duplicates of another row
print("Duplicate rows:", df.duplicated().sum())

# Remove duplicate rows so the same example is not counted twice
df = df.drop_duplicates()

# New size of the data: (rows, columns)
print("Shape after cleaning:", df.shape)

Duplicate rows: 74
Shape after cleaning: (12610, 23)


### 3.2 Separating features (X) and target (Y)

In [95]:
# X = all input columns (features), y = the column we want to predict
X = df.drop(columns=['Y'])
y = df['Y']

# Split feature names into text (categorical) and number (numeric) columns
# because they need different preprocessing
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numeric_cols = X.select_dtypes(exclude='object').columns.tolist()

print("Categorical columns:", len(categorical_cols))
print("Numeric columns:", len(numeric_cols))

Categorical columns: 17
Numeric columns: 5


### 3.3 Train/test split

In [96]:
from sklearn.model_selection import train_test_split

# My own fixed seed so results are the same every time I run
RANDOM_STATE = 29

# Keep 80% of rows for training and 20% for testing
# stratify=y keeps the same accept/reject ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

Training rows: 10088
Test rows: 2522


### 3.4 Save test data for the Streamlit app

In [97]:
# Combine test features and test target into one table
# The app needs both: features to predict, and Y to calculate metrics
test_data = X_test.copy()
test_data['Y'] = y_test

# Save as CSV without the row index column
test_data.to_csv('test_data.csv', index=False)

# Download the file to my computer so I can upload it to GitHub
from google.colab import files
files.download('test_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 3.5 Preprocessing pipeline (missing values, encoding, scaling)

In [98]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Steps for text columns:
# 1. Fill missing values with the most common value in that column
# 2. Turn each category into its own 0/1 column (one-hot encoding)
#    handle_unknown='ignore' avoids errors if the app sees a new category
categorical_steps = Pipeline([
    ('fill_missing', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Steps for number columns: scale them to a similar range
# (important for KNN, SVM and Logistic Regression)
numeric_steps = Pipeline([
    ('scale', StandardScaler())
])

# Apply the right steps to the right columns
preprocessor = ColumnTransformer([
    ('cat', categorical_steps, categorical_cols),
    ('num', numeric_steps, numeric_cols)
])

# Test the recipe on training data to check it works
X_train_ready = preprocessor.fit_transform(X_train)
print("Shape after preprocessing:", X_train_ready.shape)

Shape after preprocessing: (10088, 107)


## 4. Evaluation Metrics Function

In [99]:
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

def evaluate_model(model, X_test, y_test):
    """Return the 6 required metrics for a trained model on the test data."""
    # Predicted class for each row (0 = reject, 1 = accept)
    y_pred = model.predict(X_test)

    # Predicted probability of class 1 (accept) – needed for AUC
    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        'Accuracy': accuracy_score(y_test, y_pred),      # % of correct predictions
        'AUC': roc_auc_score(y_test, y_prob),            # how well the model ranks accepters above rejecters
        'Precision': precision_score(y_test, y_pred),    # of predicted accepts, how many were real accepts
        'Recall': recall_score(y_test, y_pred),          # of real accepts, how many we caught
        'F1': f1_score(y_test, y_pred),                  # balance of precision and recall
        'MCC': matthews_corrcoef(y_test, y_pred)         # overall quality, -1 (worst) to +1 (perfect)
    }

# Empty dictionary to collect results of every model
results = {}